# 01 · MCP 연결 · 인증 · 툴 목록 확인

**파이프라인 순서 1/4** — Teams / Mail MCP 서버에 로그인하고 연결을 확인합니다.

이 노트북이 하는 일:
1. `.env` 설정(테넌트, 클라이언트 ID, AUTH_MODE, LLM)을 확인합니다.
2. 두 MCP 소스(Mail / Teams)에 대해 **디바이스 코드 로그인**을 1회 수행합니다.
   - 로그인 토큰은 `.token_cache.json`에 저장되어, 이후 노트북과 **웹앱이 조용히 재사용**합니다.
3. 각 MCP 서버가 노출하는 **툴 목록**을 가져와 연결을 검증합니다.

> **사전 준비**: 프로젝트 루트의 `.env.example`을 `.env`로 복사하고 `TENANT_ID`, `CLIENT_ID`,
> LLM 설정을 채우세요(MCP URL은 비워두면 `TENANT_ID`로 자동 구성). 기본
> `AUTH_MODE=device_code`를 권장합니다 — Work IQ Mail/Teams MCP는 위임(delegated) 스코프를 사용합니다.
>
> ⚠️ device-code 로그인이 되려면 Entra 앱 등록에 **"Allow public client flows" = Yes**가
> 설정되어 있고, Agent Tools 리소스에 대한 위임 권한이 **관리자 동의**되어 있어야 합니다.
> 앱 등록·권한·동의 절차와 문제 해결은 **`../SETUP.md`** 를 참고하세요.

In [ ]:
# --- Bootstrap: locate llmwiki-pipeline/app and import the shared package ---
import sys
from pathlib import Path

def _find_app_dir() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        for cand in (base / 'app', base / 'llmwiki-pipeline' / 'app'):
            if (cand / 'pipeline' / '__init__.py').is_file():
                return cand
    raise RuntimeError('Could not locate llmwiki-pipeline/app — run this from inside the project.')

APP_DIR = _find_app_dir()
if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))
print('app dir on sys.path:', APP_DIR)

## 1. 설정 확인

In [ ]:
from pipeline import config

print('TENANT_ID    :', config.TENANT_ID)
print('CLIENT_ID set:', bool(config.config.client_id))
print('AUTH_MODE    :', config.config.auth_mode)
print('LLM provider :', config.llm_provider())
print('WIKI_DIR     :', config.wiki_path())
print('token cache  :', config.token_cache_file())
print()
for key, src in config.SOURCES.items():
    print(f'  [{key}] {src.label} -> {src.mcp_server_url}')

# CLIENT_ID가 없으면 여기서 친절히 알려줍니다.
config.assert_client_id()

## 2. 로그인 (device-code)

아래 셀을 실행하면 각 소스별로 로그인 안내 메시지(`https://microsoft.com/devicelogin` + 코드)가
출력됩니다. 브라우저에서 코드를 입력해 로그인하면 토큰이 캐시에 저장됩니다.
이미 캐시된 토큰이 있으면 로그인 없이 즉시 통과합니다.

> `AUTH_MODE=client_credentials`(앱 전용)로 설정한 경우 로그인 창 없이 바로 토큰을 발급합니다.

In [ ]:
from pipeline import auth, config

tokens = {}
for key in config.SOURCE_KEYS:
    print(f'\n=== 로그인: {key} ({config.SOURCES[key].label}) ===')
    try:
        tokens[key] = auth.ensure_access_token(key)
        print(f'  OK — {key} 토큰 확보 (len={len(tokens[key])})')
    except Exception as exc:
        print(f'  실패 — {key}: {exc}')

print('\n확보된 소스:', list(tokens.keys()))

## 3. 연결 검증 — 툴 목록 조회

각 MCP 서버에 연결해 사용 가능한 툴(함수) 목록을 가져옵니다.
여기서 툴 이름을 확인해 두면 노트북 02(샘플 전송)·03(직접 호출)에서 활용할 수 있습니다.

In [ ]:
from pipeline import mcp_client

tools_by_source = {}
for key, token in tokens.items():
    try:
        tools = await mcp_client.list_tools(token, key)
        tools_by_source[key] = tools
        print(f'\n=== {key}: {len(tools)}개 툴 ===')
        for t in tools:
            desc = (getattr(t, 'description', '') or '').split(chr(10))[0][:80]
            print(f'  - {t.name}: {desc}')
    except Exception as exc:
        print(f'{key} 툴 목록 조회 실패: {exc}')

특정 툴의 입력 스키마(파라미터)를 자세히 보고 싶다면:

In [ ]:
import json

# 예: 첫 번째 소스의 처음 3개 툴 스키마 출력 (원하는 툴 이름으로 바꿔서 확인하세요)
for key, tools in tools_by_source.items():
    print(f'\n########## {key} ##########')
    for t in tools[:3]:
        print(f'\n### {t.name}')
        print(json.dumps(mcp_client.tool_input_schema(t), ensure_ascii=False, indent=2)[:1200])

✅ 여기까지 성공했다면 인증·연결이 완료된 것입니다. `.token_cache.json`이 생성되어
이후 노트북과 웹앱(`app/main.py`)이 이 토큰을 재사용합니다. → **02_seed_sample_data**로 이동하세요.